In [0]:
clean_sample = spark.table("workspace.default.clean_sample").toPandas()
print(clean_sample.shape)

(2000, 2)


In [0]:
import numpy as np
import pandas as pd

def corrupt_dataset(df, null_rate=0.05, drop_column_prob=0.3, schema_drift_prob=0.2, seed=1):
    rng = np.random.default_rng(seed)
    corrupted = df.copy()

    for col in corrupted.select_dtypes(include=['float64', 'object']).columns[:3]:
        mask = rng.random(len(corrupted)) < null_rate
        corrupted.loc[mask, col] = np.nan

    if rng.random() < drop_column_prob:
        drop_col = rng.choice(corrupted.columns)
        corrupted = corrupted.drop(columns=[drop_col])
        print(f"[corruption] dropped column: {drop_col}")

    if rng.random() < schema_drift_prob:
        cols = list(corrupted.columns)
        target = rng.choice(cols)
        corrupted = corrupted.rename(columns={target: target + "_v2"})
        print(f"[corruption] renamed column: {target} -> {target}_v2")

    return corrupted

In [0]:
from datetime import datetime, timedelta

def check_schema(df, expected_columns):
    actual = set(df.columns)
    expected = set(expected_columns)
    missing = expected - actual
    extra = actual - expected
    return {"passed": len(missing) == 0, "missing_columns": list(missing), "extra_columns": list(extra)}

def check_nulls(df, threshold=0.02):
    null_rates = df.isnull().mean()
    flagged = null_rates[null_rates > threshold]
    return {"passed": len(flagged) == 0, "null_rates": null_rates.to_dict(), "flagged_columns": flagged.to_dict()}

def check_freshness(df, timestamp_col, max_age_hours=6):
    age_hours = (datetime.now() - pd.to_datetime(df[timestamp_col])).dt.total_seconds() / 3600
    stale = age_hours > max_age_hours
    return {"passed": stale.sum() == 0, "stale_row_count": int(stale.sum()), "stale_row_pct": float(stale.mean())}

def calculate_trust_score(schema_result, null_result, freshness_result):
    score = 100
    if not schema_result['passed']:
        score -= 30
    null_rate_avg = sum(null_result['null_rates'].values()) / len(null_result['null_rates'])
    score -= min(null_rate_avg * 100, 30)
    if not freshness_result['passed']:
        score -= 20
    return max(round(score, 1), 0)

expected_cols = list(clean_sample.columns)
print("Expected columns:", expected_cols)

Expected columns: ['Document', 'Topic_group']


In [0]:
results_log = []

for seed in range(1, 121):
    sample = corrupt_dataset(clean_sample, seed=seed)

    rng = np.random.default_rng(seed)
    max_hours = rng.choice([2, 6, 12, 48])
    sample = sample.copy()
    sample['load_timestamp'] = [
        datetime.now() - timedelta(hours=float(rng.uniform(0, max_hours)))
        for _ in range(len(sample))
    ]

    schema_result = check_schema(sample, expected_cols)
    null_result = check_nulls(sample, threshold=0.06)
    freshness_result = check_freshness(sample, 'load_timestamp')
    trust = calculate_trust_score(schema_result, null_result, freshness_result)

    results_log.append({
        "pipeline_id": f"pipeline_{seed:03d}",
        "schema_passed": schema_result['passed'],
        "null_check_passed": null_result['passed'],
        "freshness_passed": freshness_result['passed'],
        "trust_score": trust,
    })

pipeline_results = pd.DataFrame(results_log)
print(pipeline_results['trust_score'].describe())
print("Schema failures:", (~pipeline_results['schema_passed']).sum())
print("Null-check failures:", (~pipeline_results['null_check_passed']).sum())
print("Freshness failures:", (~pipeline_results['freshness_passed']).sum())

[corruption] renamed column: Document -> Document_v2
[corruption] dropped column: Document
[corruption] renamed column: Topic_group -> Topic_group_v2
[corruption] dropped column: Topic_group
[corruption] dropped column: Document
[corruption] dropped column: Topic_group
[corruption] renamed column: Topic_group -> Topic_group_v2
[corruption] dropped column: Topic_group
[corruption] dropped column: Topic_group
[corruption] renamed column: Document -> Document_v2
[corruption] dropped column: Document
[corruption] dropped column: Document
[corruption] dropped column: Document
[corruption] dropped column: Topic_group
[corruption] dropped column: Document
[corruption] dropped column: Topic_group
[corruption] dropped column: Document
[corruption] renamed column: Document -> Document_v2
[corruption] renamed column: Document -> Document_v2
[corruption] renamed column: Document -> Document_v2
[corruption] renamed column: Document -> Document_v2
[corruption] renamed column: Topic_group -> Topic_gr

In [0]:
print("Databricks column order:", clean_sample.columns.tolist())
print("Databricks dtypes:\n", clean_sample.dtypes)

Databricks column order: ['Document', 'Topic_group']
Databricks dtypes:
 Document       object
Topic_group    object
dtype: object


In [0]:
print("Expected columns used:", expected_cols)

Expected columns used: ['Document', 'Topic_group']


In [0]:
print("Databricks clean_sample shape:", clean_sample.shape)

Databricks clean_sample shape: (2000, 2)


In [0]:
import numpy, pandas
print("Databricks numpy version:", numpy.__version__)
print("Databricks pandas version:", pandas.__version__)

Databricks numpy version: 2.1.3
Databricks pandas version: 2.2.3
